# Code Comment Generator

This project will use a frontier model to build a code comment generator. It will take in code from any language, add proper docstrings and comments.

In [64]:
# imports

import os
import re
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [66]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

In [67]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"

anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)

In [70]:
CODE = """
    def add(a,b):
        return a + b
"""

In [72]:
# This is for when displaying in Gradio

models = ["gpt-5", "claude-sonnet-4-5" ]

clients = { "gpt-5": openai, "claude-sonnet-4-5": anthropic_client }


In [73]:
def generate_comment(code, model_name):
    prompt = f"""
I want you to act as a senior level code reviewer. I will provide you with a code snippet, and I want you to generate docstrings and comments for the code.
The docstrings should provide context of the arguments, return values, and any exceptions that may be raised.
The comment(s) must be necessary to explain any complex logic in the code. Ideally, the comments should be concise and to the point.
It is not compulsory to add comment if the code is easy to understand but it is compulsory to add docstrings.
The dosctrings must be simple and short.

Here is the code snippet:
{code}

The output should be the code snippet with the docstrings and comments added.
The output should be in a code block and should not contain any additional text or explanations.

Do not, under any cicumstances add any code only add docstrings and comments to the code provided. Do not add any additional text or explanations.
"""
    client = clients[model_name]

    if model_name.startswith("claude"):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1500,
        )
        return response.choices[0].message.content

    request = {
        "model": model_name,
        "input": prompt,
        "max_output_tokens": 1500,
    }

    if model_name.startswith("gpt-5"):
        request["reasoning"] = {"effort": "minimal"}

    response = client.responses.create(**request)
    return response.output_text
    

In [74]:
gpt_output = generate_comment(CODE, "claude-sonnet-4-5")

In [ ]:
Markdown(gpt_output)

In [76]:
def extract_code_block(text):
    if not text:
        return ""

    match = re.search(r"```(?:\w+)?\n([\s\S]*?)\n```", text.strip())
    if match:
        return match.group(1).strip()

    return text.strip()


def comment_code(code, model_name):
    if not code or not code.strip():
        return ""

    commented_code = generate_comment(code, model_name)
    return extract_code_block(commented_code)


APP_CSS = """
.controls {
    align-items: center;
    justify-content: center;
    gap: 12px;
}

.comment-btn {
    min-width: 220px;
}
"""

In [ ]:
with gr.Blocks(css=APP_CSS, theme=gr.themes.Monochrome(), title="Code Comment Generator") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            source_code = gr.Code(
                label="Code (original)",
                value=CODE.strip(),
                lines=26
            )
        with gr.Column(scale=6):
            commented_code = gr.Code(
                label="Code with docstrings and comments",
                value="",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0], show_label=False)
        add_comments = gr.Button("Add docstrings + comments", elem_classes=["comment-btn"])
        clear = gr.Button("Clear")

    add_comments.click(fn=comment_code, inputs=[source_code, model], outputs=[commented_code])
    clear.click(fn=lambda: ("", ""), inputs=[], outputs=[source_code, commented_code])

ui.launch(inbrowser=True)
